In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

import os

from selenium.webdriver.common.action_chains import ActionChains



    

In [ ]:
# %%
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CA OSFI' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.2")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





In [ ]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

actionChains = ActionChains(driver)





In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

    'CA OSFI 1': 'Domestic Banks', 

	'CA OSFI 2': 'Foreign Banks', 

	'CA OSFI 3': 'Foreign Bank Branches - Full Service', 

	'CA OSFI 4': 'Foreign Bank Branches - Lending', 

	'CA OSFI 5': 'Trust Companies', 

	'CA OSFI 6': 'Loan Companies', 

	'CA OSFI 9': 'Canadian Life Insurance Companies', 

	'CA OSFI 10': 'Foreign Life Insurance Companies', 

	'CA OSFI 11': 'Canadian Fraternal Benefit Societies', 

	'CA OSFI 12': 'Foreign Fraternal Benefit Societies', 

	'CA OSFI 13': 'Canadian Property & Casualty Insurance Companies', 

    

	'CA OSFI 14': 'Foreign Property & Casualty Insurance Companies', 

	'CA OSFI 15': 'Foreign Bank Representative Offices',

	'CA OSFI 16': 'Canadian Mortgage Insurers', 

    }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')




    

In [ ]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def ListItems(btn_id, level) :

    items = {}

    btn = driver.find_element(By.ID, btn_id)

    actionChains.move_to_element(btn).perform()

    sleep(1)

    btn.click()

    sleep(2)

    soup=BeautifulSoup(driver.page_source.replace('<br>','****').replace('<br/>','****'),"html.parser")

    visibleGroup=soup.find_all('div', {"class": "visibleGroup"})

    buttons = visibleGroup[level].find_all('div', {"role": "button"})

    Id_Last_Iten = buttons[len(buttons)-1].find('g').attrs['id']



    while True : 

        for btn_ in buttons : 

            if btn_.find('g').attrs['id'] not in items :

                items[btn_.find('g').attrs['id']]=btn_.text



        scrolling = driver.find_element(By.ID, Id_Last_Iten)

        actionChains.move_to_element(scrolling).perform()

        sleep(1)

        soup=BeautifulSoup(driver.page_source.replace('<br>','****').replace('<br/>','****'),"html.parser")

        visibleGroup=soup.find_all('div', {"class": "visibleGroup"})

        buttons = visibleGroup[level].find_all('div', {"role": "button"})

        lastID = buttons[len(buttons)-1].find('g').attrs['id']



        if lastID == Id_Last_Iten :

            break

        else :

            Id_Last_Iten = lastID

            

    return items



# %%


In [ ]:

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://www.osfi-bsif.gc.ca/en/supervision/who-we-regulate')

sleep(1)

for time in range(20):

    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.DOWN)



sleep(15)

element_frames = driver.find_elements(By.TAG_NAME, "iframe")

driver.switch_to.frame(element_frames[0])

sleep(5)

soup=BeautifulSoup(driver.page_source.replace('<br>','****').replace('<br/>','****'),"html.parser")

visibleGroup=soup.find_all('div', {"class": "visibleGroup"})

buttons_L1 = visibleGroup[0].find_all('div', {"role": "button"})



for btn_L1 in buttons_L1 :

    items_L2 = ListItems(btn_L1.find('g').attrs['id'], 1)

    list_nom_L2 = list(reversed(list(items_L2.keys())))

    print(f"[INFO] :- {btn_L1.text} " )

    

    for i, btn_L2 in enumerate(list_nom_L2) :

        items_L3 = ListItems(btn_L2, 2)

        list_nom_L3 = list(reversed(list(items_L3.keys())))

        print(f"[INFO] :    - {items_L2[btn_L2]} " )



        if i+1 < len(list_nom_L2) :

            scrolling = driver.find_element(By.ID, list_nom_L2[i+1])

            actionChains.move_to_element(scrolling).perform()

            sleep(1)

       

        for j, btn_L3 in enumerate(list_nom_L3) : 

            Typology = items_L3[btn_L3].split('(')[0].strip()

            campany_detail = {}

            try:

                reg = list(regdict.keys())[list(regdict.values()).index(Typology)]

            except:

                reg= ''

            

            driver.find_element(By.ID, btn_L3).click()

            sleep(2)



            for m in range(2):



                soup=BeautifulSoup(driver.page_source.replace('<br>','****').replace('<br/>','****'),"html.parser")

                all_row=soup.find('div', {"class": "mid-viewport"}).find_all('div', {"role": "row"})

                Last_company = all_row[-1].find_all('div', {"role": "gridcell"})[1].text

                if Last_company.find('****')!= -1 :

                    Last_company =  Last_company[:Last_company.index('****')]

    

                while True : 

                    for row in all_row :

                        campany_name =  row.find_all('div', {"role": "gridcell"})[1].text



                        if reg == 'CA OSFI 3' or reg == 'CA OSFI 4':

                            address =  row.find_all('div', {"role": "gridcell"})[3].text

                        else:

                            address =  row.find_all('div', {"role": "gridcell"})[2].text



                        if campany_name.find('****')!= -1 :

                            address = campany_name[campany_name.index('****')+4:]

                            campany_name =  campany_name[:campany_name.index('****')]

                        if campany_name not in campany_detail :

                            campany_detail[campany_name]=address



                    scrolling = driver.find_elements(By.CLASS_NAME, 'pivotTableCellWrap')[-3].send_keys(Keys.DOWN)

                    sleep(1)

                    soup=BeautifulSoup(driver.page_source.replace('<br>','****').replace('<br/>','****'),"html.parser")

                    all_row=soup.find('div', {"class": "mid-viewport"}).find_all('div', {"role": "row"})

                    New_last_company = all_row[-1].find_all('div', {"role": "gridcell"})[1].text

                    if New_last_company.find('****')!= -1 :

                        New_last_company =  New_last_company[:New_last_company.index('****')]



                    if New_last_company == Last_company :

                        break

                    else :

                        Last_company = New_last_company

                    

                driver.find_element(By.XPATH, '//*/div[@role="columnheader" and @aria-colindex="2"]' ).click() # changer l'oredre de tri de la colonne

                sleep(2)



            if Typology in regdict.values() :



                for name in campany_detail : 

                    address_ = campany_detail[name].split('****')

                    if len(address_) == 6 :

                        sqldict['Address_2'].append(address_[3])

                    else :

                        sqldict['Address_2'].append('')



                    if len(address_) > 4 :

                        address_  = address_ [2:]



                    sqldict['Name'].append(name)

                    sqldict['Typology'].append(Typology)

                    sqldict['Address_1'].append(address_[0])

                    sqldict['Zip'].append(address_[-1])

                    sqldict['City'].append(address_[-2].split(',')[0])

                    sqldict["Cntry"].append("CA")

                    sqldict["RegulationType"].append("Regulated")

                    sqldict['ListProcessDate'].append(processdate)

                    sqldict['RegCtry'].append(reg.split(' ')[0])

                    sqldict['RegCode'].append(reg.split(' ')[1])

                    sqldict['ListCode'].append(reg.split(' ')[-1])



                sqldict = bourange_same_length_array(sqldict)



                print(f"[INFO] :        - {reg} : ({len(campany_detail)}) campany | {items_L3[btn_L3]}:" )



                # if int(items_L3[btn_L3].split('(')[-1][:-1]) != len(campany_detail) :

                #     raise Exception(f'[ERROR] : Valeur differente | "{items_L3[btn_L3]}" ')

                

                if j+1 < len(list_nom_L3) :

                    try:

                        scrolling = driver.find_element(By.ID, list_nom_L3[j+1])

                        actionChains.move_to_element(scrolling).perform()

                        sleep(1)

                    except:

                        print(f"[ERROR] : ID = {list_nom_L3[j+1]}")

                        for time in range(10):

                            driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.UP)




In [ ]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)